> 在线讲义：[第 57 章 · 训练状态的分片、重计算与卸载：显存不够时的三条路](https://wikieden.github.io/ai-infra-from-constraints/lessons/57-state-sharding.html) · 本书《从约束推导 AI Infra》

# 第 57 章 · 训练状态的分片、重计算与卸载：显存不够时的三条路

配套讲义：仓库里 `ai-infra/notes/lessons/57-state-sharding.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



ZeRO 各阶段每卡容量、门控乘积重算、CPU / GPU 转换梯度的带宽分界与 RTX 4090 的容量方案，取自 AII 第 10 章 §10.2。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
MiB, GiB = 2**20, 2**30
N = 8_190_735_360

**函数 `zero_bytes`**：每卡常驻状态：M0 = 16N，M1 = 4N + 12N/d，M2 = 2N + 14N/d，M3 = 16N/d。

In [ ]:
def zero_bytes(stage: int, d: int, n: int = N) -> float:
    """每卡常驻状态：M0 = 16N，M1 = 4N + 12N/d，M2 = 2N + 14N/d，M3 = 16N/d。"""
    return {0: 16 * n, 1: 4 * n + 12 * n / d, 2: 2 * n + 14 * n / d, 3: 16 * n / d}[stage]

**函数 `recompute_saving_mib`**：每层省下 h（[t, 12288] FP32）和两个 [t, 4096] 归一化输出。

In [ ]:
def recompute_saving_mib(tokens: int = 128, layers: int = 9, mbs: int = 1) -> float:
    """每层省下 h（[t, 12288] FP32）和两个 [t, 4096] 归一化输出。"""
    per_layer = tokens * (12288 + 2 * 4096) * 4 / MiB
    return per_layer * layers * mbs

**函数 `wait_s`**：T_wait = max(0, V/B − W)。

In [ ]:
def wait_s(V: float, B: float, W: float) -> float:
    """T_wait = max(0, V/B − W)。"""
    return max(0.0, V / B - W)

**函数 `cast_paths_ms`**：(CPU 转换, GPU 转换) 的串行时间：T_c = G/B + 3G/C_c，T_g = 3G/C_g + 2G/B。

In [ ]:
def cast_paths_ms(G: float = 96 * MiB, B: float = 32e9, Cc: float = 307.2e9, Cg: float = 1008e9):
    """(CPU 转换, GPU 转换) 的串行时间：T_c = G/B + 3G/C_c，T_g = 3G/C_g + 2G/B。"""
    return (G / B + 3 * G / Cc) * 1e3, (3 * G / Cg + 2 * G / B) * 1e3

**函数 `cast_breakeven`**：B* = 1 / (3 (1/C_c − 1/C_g))。

In [ ]:
def cast_breakeven(Cc: float = 307.2e9, Cg: float = 1008e9) -> float:
    """B* = 1 / (3 (1/C_c − 1/C_g))。"""
    return 1 / (3 * (1 / Cc - 1 / Cg))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：ZeRO 各阶段（d = 8）")
for st, name in enumerate(("普通 DP", "ZeRO-1", "ZeRO-2", "ZeRO-3")):
    print(f"  {name}：{zero_bytes(st, 8) / GiB:.1f} GiB/卡")
print("  FSDP2 小实验：常驻 18 → 9 MiB，峰值 39 → 30 MiB（另有约 21 MiB 执行期张量）")

In [ ]:
print("\n第二步：选择性重计算")
print(f"  h = [128, 12288] FP32 = {128 * 12288 * 4 / MiB:.0f} MiB；每层省 {recompute_saving_mib(layers=1):.0f} MiB，九层 {recompute_saving_mib():.0f} MiB")
print(f"  九层 GEMM 输入 135 → {135 - recompute_saving_mib():.0f} MiB；四个 micro-batch 等反向省 {recompute_saving_mib(mbs=4):.0f} MiB")
print(f"  重建 h 要 {128 * 12288 / 1e6:.2f} M 次乘法")

In [ ]:
print("\n第三步：卸载与转换位置（96 MiB BF16 梯度）")
for name, B in (("PCIe 4.0 x16 32 GB/s", 32e9), ("NVLink-C2C 450 GB/s", 450e9)):
    c, g = cast_paths_ms(B=B)
    print(f"  {name}：CPU 转换 {c:.2f} ms，GPU 转换 {g:.2f} ms")
print(f"  分界带宽 B* = {cast_breakeven() / 1e9:.0f} GB/s；GPU 转换峰值 {96 + 192} MiB")
print(f"  例：1 GiB 状态、32 GB/s、提前 20 ms 开始 → 等 {wait_s(GiB, 32e9, 0.02) * 1e3:.1f} ms")

In [ ]:
print("\n第四步：RTX 4090 每卡 22 GiB")
for d in (8, 16, 32, 48):
    s = zero_bytes(3, d) / GiB
    print(f"  ZeRO-3 {d:>2} 卡：常驻 {s:.1f} GiB + 10 GiB = {s + 10:.1f} GiB，余量 {22 - s - 10:+.1f}")

## 3. 代码里才看得到的

1. 转换"读 $G$ 写 $2G$"共 $3G$ 字节，转换时间按内存带宽算，不按算力。
2. `cast_breakeven` 与 $G$ 无关：两条路径的差都正比于 $G$，交点只由三种带宽决定。
3. 22 GiB 是按 24 GB = 22.35 GiB 再扣运行时得到的，余量表都基于它。

## 4. 自己改一改

- 给 `zero_bytes` 加"梯度 FP32"的版本，看 ZeRO-2 的变化。
- 把 `cast_paths_ms` 的 $G$ 换成整个模型的梯度，看 PCIe 上一步要多久。
- 用 `wait_s` 算：要让 1 GiB 状态不等，至少要提前多少毫秒开始传。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 合租的书先分参考书、再分笔记、最后连正在读的也分，要读时找室友借；
> 草稿只留关键几步，检查前现算；
> 寄存隔壁仓库要及时取回，路窄就少运；
> 房间不够，多租一间还是叠起来放，先看放不放得下，再看谁快。